# importing functions 

In [0]:
from  pyspark.sql.functions import *

# Reading data
# 

In [0]:
df = spark.read.format('parquet')\
    .option('inferSchema', True)\
    .load('abfss://bronze@carlake2003.dfs.core.windows.net/rawdata')

In [0]:
df.show()

+---------+---------+--------+--------+----------+-------+---+-----+----+--------------------+--------------------+
|Branch_ID|Dealer_ID|Model_ID| Revenue|Units_Sold|Date_ID|Day|Month|Year|          BranchName|          DealerName|
+---------+---------+--------+--------+----------+-------+---+-----+----+--------------------+--------------------+
|   BR0001|  DLR0001|  BMW-M1|13363978|         2|DT00001|  1|    1|2017|      AC Cars Motors|      AC Cars Motors|
|   BR0003|  DLR0228|Hon-M218|17376468|         3|DT00001| 10|    5|2017|      AC Cars Motors|       Deccan Motors|
|   BR0004|  DLR0208|Tat-M188| 9664767|         3|DT00002| 12|    1|2017|      AC Cars Motors|     Wiesmann Motors|
|   BR0005|  DLR0188|Hyu-M158| 5525304|         3|DT00002| 16|    9|2017|      AC Cars Motors|       Subaru Motors|
|   BR0006|  DLR0168|Ren-M128|12971088|         3|DT00003| 20|    5|2017|      AC Cars Motors|         Saab Motors|
|   BR0008|  DLR0128| Hon-M68| 7321228|         1|DT00004| 28|    4|2017

# DATA TRANSFORMATION

In [0]:
df = df.withColumn('model_category', split(col('Model_ID'), '-')[0])

In [0]:
df=df.withColumn('Revperunit', col('Revenue')/col('Units_Sold'))


In [0]:
df.groupby('year','branchname').agg(sum('Units_Sold').alias('total_unit')).sort('year','total_unit',ascending=[0,1])


DataFrame[year: int, branchname: string, total_unit: bigint]

In [0]:
df.write.format('parquet')\
    .mode('append')\
    .option('path','abfss://silver@carlake2003.dfs.core.windows.net/carsales')\
    .save()